In [23]:
import sys
import time
import math
import csv
import importlib
from pathlib import Path

import numpy as np

root = Path.cwd()
parents = [root, root.parent, root.parent.parent]
for base in parents:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

import efgp_eigenpro_py.efgp_solver as efgp_solver
import efgp_eigenpro_py.toeplitz as toeplitz
import efgp_eigenpro_py.eigenspace as eigenspace
importlib.reload(toeplitz)
importlib.reload(efgp_solver)
importlib.reload(eigenspace)
from efgp_eigenpro_py.efgp_solver import EFGPSolver, PrecomputeState
from efgp_eigenpro_py.discretization import choose_grid_params, basis_weights
from efgp_eigenpro_py.kernels import make_squared_exponential, make_matern
from efgp_eigenpro_py.nufft_ops import nufftnd1
from efgp_eigenpro_py.toeplitz import make_toeplitz_workspace
from efgp_eigenpro_py import benchmark as bm
importlib.reload(bm)

np.set_printoptions(precision=4, suppress=True)

# paper settings
SIGMA_TRUE = 0.3
VARIANCE = 1.0
LENGTHSCALE = 0.1
NU_MATERN = 0.5
NTRGS_PER_D = 60

EPS_SE_BY_DIM = {1: 1e-4, 2: 1e-4, 3: 1e-3}
EPS_MAT_BY_DIM = {1: 1e-4, 2: 1e-3, 3: 5e-3}

# use paper-like L2 scaling for Matern small nu
L2SCALED_SE = True
L2SCALED_MATERN = True

# data sizes
RUN_FULL_N = True
if RUN_FULL_N:
    N_LIST = [1000, 100000, 10000000]
else:
    N_LIST = [1000, 100000]

TOP_Q_SWEEP = [0, 4, 8, 16]
CG_MAXITER = 20000

CSV_OUTPUT_DIR = Path("results")
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH_TABLE2_TOPQ = CSV_OUTPUT_DIR / "greengard_table2_topq.csv"

# eigensolver settings (top-q preconditioner)
EIG_METHOD = "eigsh"  # "eigsh", "lobpcg", "subspace_iter"
EIG_TOL = 1
EIG_MAXITER = 5
EIG_BLOCK_SIZE = None
EIG_OVERSAMPLE = 0

# dimension sweep
DIM_LIST = [1,2,3] #[1,2,3]

# training data chunk size
TRAIN_CHUNK_SIZE = 10000000
USE_STREAMING_PRECOMPUTE = True
STREAMING_N_THRESHOLD = 5 * 10**7

# timing alignment with gp-shootout
MEAN_INCLUDE_TRAIN = True
USE_GPSHOOTOUT_BOUNDS = True

# reference mode
REF_MODE = "efgp"  # only iterative reference
COMPUTE_EEPM = False

print("RUN_FULL_N=", RUN_FULL_N, "N_LIST=", N_LIST)


RUN_FULL_N= True N_LIST= [1000, 100000, 10000000]


In [24]:
def make_wavevec(dim: int) -> np.ndarray:
    if dim == 1:
        return np.array([3.0], dtype=float)
    if dim == 2:
        v = np.array([1.0, 2.0], dtype=float)
        v = 3.0 * v / np.linalg.norm(v)
        return v
    if dim == 3:
        v = np.array([1.0, 3.0, 2.0], dtype=float)
        v = 3.0 * v / np.linalg.norm(v)
        return v
    raise ValueError("dim must be 1, 2, or 3 for this table")


def f_eval(x: np.ndarray, wavevec: np.ndarray) -> np.ndarray:
    return np.cos(2.0 * math.pi * (x @ wavevec) + 1.3)


def equispaced_grid(dim: int, n_per_d: int) -> np.ndarray:
    grid_1d = np.linspace(0.0, 1.0, n_per_d, endpoint=False)
    mesh = np.meshgrid(*([grid_1d] * dim), indexing="ij")
    return np.stack([g.reshape(-1) for g in mesh], axis=1)


def generate_train_chunks(dim: int, n: int, chunk_size: int, seed: int, sigma_true: float):
    rng = np.random.default_rng(seed)
    wavevec = make_wavevec(dim)
    produced = 0
    while produced < n:
        m = min(chunk_size, n - produced)
        x = rng.uniform(0.0, 1.0, size=(m, dim))
        phase = 2.0 * math.pi * (x @ wavevec) + 1.3
        f = np.cos(phase)
        y = f + sigma_true * rng.standard_normal(m)
        yield x, y
        produced += m


def make_train_data(
    dim: int,
    n: int,
    seed: int = 0,
    chunk_size: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if chunk_size is None:
        chunk_size = n
    wavevec = make_wavevec(dim)
    xs = []
    ys = []
    for x, y in generate_train_chunks(dim, n, chunk_size, seed, SIGMA_TRUE):
        xs.append(x)
        ys.append(y)
    return np.vstack(xs), np.concatenate(ys), wavevec


def select_shared_subset(
    dim: int,
    n_total: int,
    indices: np.ndarray,
    seed: int,
    chunk_size: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if chunk_size is None:
        chunk_size = n_total
    idx = np.asarray(indices, dtype=int)
    if idx.size == 0:
        return np.empty((0, dim)), np.empty((0,)), make_wavevec(dim)
    wavevec = make_wavevec(dim)
    x_sel = np.empty((idx.size, dim))
    y_sel = np.empty(idx.size)
    rng = np.random.default_rng(seed)
    produced = 0
    ptr = 0
    while produced < n_total and ptr < idx.size:
        m = min(chunk_size, n_total - produced)
        x = rng.uniform(0.0, 1.0, size=(m, dim))
        phase = 2.0 * math.pi * (x @ wavevec) + 1.3
        f = np.cos(phase)
        y = f + SIGMA_TRUE * rng.standard_normal(m)

        end = produced + m
        while ptr < idx.size and idx[ptr] < end:
            local = idx[ptr] - produced
            x_sel[ptr] = x[local]
            y_sel[ptr] = y[local]
            ptr += 1
        produced = end

    return x_sel, y_sel, wavevec


def generate_shared_subset_chunks(
    dim: int,
    n_total: int,
    indices: np.ndarray,
    seed: int,
    chunk_size: int,
    sigma_true: float,
):
    idx = np.asarray(indices, dtype=int)
    if idx.size == 0:
        return
    wavevec = make_wavevec(dim)
    rng = np.random.default_rng(seed)
    produced = 0
    ptr = 0
    buf_x = []
    buf_y = []

    while produced < n_total and ptr < idx.size:
        m = min(chunk_size, n_total - produced)
        x = rng.uniform(0.0, 1.0, size=(m, dim))
        phase = 2.0 * math.pi * (x @ wavevec) + 1.3
        f = np.cos(phase)
        y = f + sigma_true * rng.standard_normal(m)

        end = produced + m
        while ptr < idx.size and idx[ptr] < end:
            local = idx[ptr] - produced
            buf_x.append(x[local])
            buf_y.append(y[local])
            if len(buf_y) >= chunk_size:
                yield np.asarray(buf_x), np.asarray(buf_y)
                buf_x = []
                buf_y = []
            ptr += 1
        produced = end

    if buf_y:
        yield np.asarray(buf_x), np.asarray(buf_y)


def make_targets(dim: int, n_per_d: int, seed: int = 0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    xtrgs = equispaced_grid(dim, n_per_d)
    wavevec = make_wavevec(dim)
    ftrgs = f_eval(xtrgs, wavevec)
    ytrgs_test = ftrgs + SIGMA_TRUE * rng.standard_normal(xtrgs.shape[0])
    return xtrgs, ytrgs_test, ftrgs


def rms(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))


In [25]:
def precompute_pure_efgp(
    solver: EFGPSolver,
    x: np.ndarray,
    y: np.ndarray,
    eps: float,
    l2scaled: bool,
    x_trg: np.ndarray | None = None,
):
    t0 = time.perf_counter()
    x = solver._ensure_2d(x)
    x0 = np.min(x, axis=0)
    x1 = np.max(x, axis=0)
    if USE_GPSHOOTOUT_BOUNDS and solver.kernel.dim == 1:
        if x_trg is not None and np.asarray(x_trg).size > 0:
            x_trg = solver._ensure_2d(x_trg)
            x0 = np.minimum(x0, np.min(x_trg, axis=0))
            x1 = np.maximum(x1, np.max(x_trg, axis=0))
    x_center = (x0 + x1) / 2.0
    L = float(np.max(x1 - x0))
    grid = choose_grid_params(solver.kernel, eps, L, l2scaled=l2scaled)
    tphx = solver._scale_from_center(x, x_center, grid.h)
    weights = np.ascontiguousarray(basis_weights(solver.kernel, grid.xis, grid.h).reshape(-1))

    dim = solver.kernel.dim
    nufft_eps = eps / 10.0
    c = np.ones(x.shape[0], dtype=complex)
    ms_xtx = 2 * grid.mtot - 1

    t1 = time.perf_counter()
    XtXcol = nufftnd1(tphx, c, ms_xtx, nufft_eps, -1)
    XtXcol = XtXcol.reshape((ms_xtx,) * dim)
    t2 = time.perf_counter()
    Gf = np.fft.fftn(XtXcol)
    t3 = time.perf_counter()
    rhs = nufftnd1(tphx, y, grid.mtot, nufft_eps, -1)
    t4 = time.perf_counter()
    rhs = np.ascontiguousarray(rhs.reshape(-1))
    np.multiply(weights, rhs, out=rhs)

    Gf = np.ascontiguousarray(Gf)
    state = PrecomputeState(
        grid=grid,
        weights=weights,
        Gf=Gf,
        rhs=rhs,
        x_center=x_center,
    )
    t5 = time.perf_counter()

    times = {
        "time_nufft_v": t2 - t1,
        "time_fft_embed": t3 - t2,
        "time_nufft_rhs": t4 - t3,
        "time_precompute_total": t5 - t0,
    }
    return state, times


def run_efgp_once(dim, kernel, eps, l2scaled, x_train, y_train, x_trg, y_trg_test, cg_tol, maxiter):
    reg_lambda = SIGMA_TRUE ** 2
    nufft_eps = eps / 10.0
    solver = EFGPSolver(kernel, reg_lambda=reg_lambda, eps=eps, nufft_tol=nufft_eps)

    state, pre_times = precompute_pure_efgp(solver, x_train, y_train, eps, l2scaled, x_trg=x_trg)

    matvec = lambda v: solver._apply_A(state, v)
    t_s0 = time.perf_counter()
    beta, it, relres, _hist = bm.pcg_solve(matvec, state.rhs, tol=cg_tol, maxiter=maxiter, precond=None, store_history=False)
    t_s1 = time.perf_counter()

    x_eval = x_trg
    if MEAN_INCLUDE_TRAIN:
        x_eval = np.vstack([x_train, x_trg])
    t_m0 = time.perf_counter()
    yhat_all = solver.predict(x_eval, beta, state)
    t_m1 = time.perf_counter()

    yhat = yhat_all[-x_trg.shape[0]:] if MEAN_INCLUDE_TRAIN else yhat_all
    err_rms = rms(y_trg_test, yhat)

    info = {
        "state": state,
        "pre_times": pre_times,
        "solve_time": t_s1 - t_s0,
        "mean_time": t_m1 - t_m0,
        "iters": it,
        "relres": relres,
        "m": (state.grid.mtot - 1) // 2,
    }
    return yhat, err_rms, info


def run_efgp_once_streaming(
    dim,
    kernel,
    eps,
    l2scaled,
    n_train,
    seed,
    x_trg,
    y_trg_test,
    cg_tol,
    maxiter,
    chunk_size,
    shared_indices=None,
    shared_total=None,
    x_train_cached=None,
):
    reg_lambda = SIGMA_TRUE ** 2
    nufft_eps = eps / 10.0
    solver = EFGPSolver(kernel, reg_lambda=reg_lambda, eps=eps, nufft_tol=nufft_eps)

    if shared_indices is None:
        chunk_iter = generate_train_chunks(dim, n_train, chunk_size, seed, SIGMA_TRUE)
    else:
        if shared_total is None:
            raise ValueError("shared_total is required when shared_indices is provided")
        chunk_iter = generate_shared_subset_chunks(
            dim,
            shared_total,
            shared_indices,
            seed,
            chunk_size,
            SIGMA_TRUE,
        )

    if USE_GPSHOOTOUT_BOUNDS:
        if x_train_cached is None:
            raise ValueError("x_train_cached is required when USE_GPSHOOTOUT_BOUNDS=True in streaming mode")
        x_min = np.min(x_train_cached, axis=0)
        x_max = np.max(x_train_cached, axis=0)
        if dim == 1:
            x_min = np.minimum(x_min, np.min(x_trg, axis=0))
            x_max = np.maximum(x_max, np.max(x_trg, axis=0))
        bounds = (x_min, x_max)
    else:
        bounds = (np.zeros(dim), np.ones(dim))

    t_p0 = time.perf_counter()
    state = solver.precompute_streaming(chunk_iter, n_total=n_train, x_bounds=bounds)
    t_p1 = time.perf_counter()

    matvec = lambda v: solver._apply_A(state, v)
    t_s0 = time.perf_counter()
    beta, it, relres, _hist = bm.pcg_solve(matvec, state.rhs, tol=cg_tol, maxiter=maxiter, precond=None, store_history=False)
    t_s1 = time.perf_counter()

    x_eval = x_trg
    if MEAN_INCLUDE_TRAIN:
        if x_train_cached is None:
            raise ValueError("x_train_cached is required when MEAN_INCLUDE_TRAIN=True in streaming mode")
        x_eval = np.vstack([x_train_cached, x_trg])
    t_m0 = time.perf_counter()
    yhat_all = solver.predict(x_eval, beta, state)
    t_m1 = time.perf_counter()

    yhat = yhat_all[-x_trg.shape[0]:] if MEAN_INCLUDE_TRAIN else yhat_all
    err_rms = rms(y_trg_test, yhat)

    info = {
        "state": state,
        "pre_times": {"time_precompute_total": t_p1 - t_p0},
        "solve_time": t_s1 - t_s0,
        "mean_time": t_m1 - t_m0,
        "iters": it,
        "relres": relres,
        "m": (state.grid.mtot - 1) // 2,
    }
    return yhat, err_rms, info


def run_reference(dim, kernel, eps_ref, l2scaled, x_train, y_train, x_trg, y_trg_test, cg_tol, maxiter):
    if REF_MODE != "efgp":
        raise ValueError("only iterative reference is supported in this notebook")

    yhat_ref, _err_rms, _info = run_efgp_once(
        dim,
        kernel,
        eps_ref,
        l2scaled,
        x_train,
        y_train,
        x_trg,
        y_trg_test,
        cg_tol,
        maxiter,
    )
    return yhat_ref


def run_reference_streaming(
    dim,
    kernel,
    eps_ref,
    l2scaled,
    n_train,
    seed,
    x_trg,
    y_trg_test,
    cg_tol,
    maxiter,
    chunk_size,
    shared_indices=None,
    shared_total=None,
    x_train_cached=None,
):
    if REF_MODE != "efgp":
        raise ValueError("only iterative reference is supported in this notebook")

    yhat_ref, _err_rms, _info = run_efgp_once_streaming(
        dim,
        kernel,
        eps_ref,
        l2scaled,
        n_train,
        seed,
        x_trg,
        y_trg_test,
        cg_tol,
        maxiter,
        chunk_size,
        shared_indices=shared_indices,
        shared_total=shared_total,
        x_train_cached=x_train_cached,
    )
    return yhat_ref


In [26]:
USE_SHARED_DATASET = True


def get_eps_and_l2scaled(kernel_name: str, dim: int) -> tuple[float, bool]:
    if kernel_name == "se":
        return EPS_SE_BY_DIM[dim], L2SCALED_SE
    if kernel_name == "matern":
        return EPS_MAT_BY_DIM[dim], L2SCALED_MATERN
    raise ValueError(kernel_name)


def print_table_header():
    print("N d kernel eps m pre_s eig_s precond_s solve_s mean_s total_s iters eepm rms rmse_ex")


def print_table_row(N, dim, kernel_label, eps, m, pre, eig, precond, solve, mean, total, iters, eepm, rms_err, rmse_ex):
    print(
        f"{N:>8d} {dim:>1d} {kernel_label:>7s} {eps:>8.1e} {m:>4d} "
        f"{pre:>8.3f} {eig:>8.3f} {precond:>8.3f} {solve:>8.3f} {mean:>8.3f} {total:>8.3f} {iters:>6d} "
        f"{eepm:>9.2e} {rms_err:>9.2e} {rmse_ex:>9.2e}"
    )


results = []
cache = {}

print_table_header()

for kernel_name in ["se", "matern"]:
    for dim in DIM_LIST:
        eps, l2scaled = get_eps_and_l2scaled(kernel_name, dim)
        if kernel_name == "se":
            kernel = make_squared_exponential(lengthscale=LENGTHSCALE, dim=dim, variance=VARIANCE)
            kernel_label = "SE"
        else:
            kernel = make_matern(lengthscale=LENGTHSCALE, dim=dim, nu=NU_MATERN, variance=VARIANCE)
            kernel_label = "Mat12"

        n_list = list(N_LIST)
        n_max = max(n_list)

        if USE_SHARED_DATASET:
            shared_seed = dim
        else:
            shared_seed = None

        x_trg, y_trg_test, _ftrgs = make_targets(dim, NTRGS_PER_D, seed=dim + 17)

        for N in n_list:
            cg_tol = eps
            inds = None
            x_train = None
            y_train = None
            use_streaming = USE_STREAMING_PRECOMPUTE and N >= STREAMING_N_THRESHOLD

            if use_streaming:
                if USE_SHARED_DATASET:
                    inds = (np.arange(1, N + 1) * (n_max / N) - 1).astype(int)
                    x_train, y_train, wavevec = select_shared_subset(
                        dim,
                        n_max,
                        inds,
                        seed=shared_seed,
                        chunk_size=TRAIN_CHUNK_SIZE,
                    )
                    yhat, err_rms, info = run_efgp_once_streaming(
                        dim,
                        kernel,
                        eps,
                        l2scaled,
                        n_train=N,
                        seed=shared_seed,
                        x_trg=x_trg,
                        y_trg_test=y_trg_test,
                        cg_tol=cg_tol,
                        maxiter=CG_MAXITER,
                        chunk_size=TRAIN_CHUNK_SIZE,
                        shared_indices=inds,
                        shared_total=n_max,
                        x_train_cached=x_train,
                    )
                else:
                    x_train, y_train, wavevec = make_train_data(
                        dim,
                        N,
                        seed=dim + N,
                        chunk_size=TRAIN_CHUNK_SIZE,
                    )
                    yhat, err_rms, info = run_efgp_once_streaming(
                        dim,
                        kernel,
                        eps,
                        l2scaled,
                        n_train=N,
                        seed=dim + N,
                        x_trg=x_trg,
                        y_trg_test=y_trg_test,
                        cg_tol=cg_tol,
                        maxiter=CG_MAXITER,
                        chunk_size=TRAIN_CHUNK_SIZE,
                        x_train_cached=x_train,
                    )
            else:
                if USE_SHARED_DATASET:
                    inds = (np.arange(1, N + 1) * (n_max / N) - 1).astype(int)
                    x_train, y_train, wavevec = select_shared_subset(
                        dim,
                        n_max,
                        inds,
                        seed=shared_seed,
                        chunk_size=TRAIN_CHUNK_SIZE,
                    )
                else:
                    x_train, y_train, wavevec = make_train_data(
                        dim,
                        N,
                        seed=dim + N,
                        chunk_size=TRAIN_CHUNK_SIZE,
                    )

                yhat, err_rms, info = run_efgp_once(
                    dim, kernel, eps, l2scaled, x_train, y_train, x_trg, y_trg_test, cg_tol, CG_MAXITER
                )

            if COMPUTE_EEPM:
                eps_ref = eps / 10.0
                if use_streaming:
                    if USE_SHARED_DATASET:
                        yhat_ref = run_reference_streaming(
                            dim,
                            kernel,
                            eps_ref,
                            l2scaled,
                            n_train=N,
                            seed=shared_seed,
                            x_trg=x_trg,
                            y_trg_test=y_trg_test,
                            cg_tol=eps_ref,
                            maxiter=CG_MAXITER,
                            chunk_size=TRAIN_CHUNK_SIZE,
                            shared_indices=inds,
                            shared_total=n_max,
                            x_train_cached=x_train,
                        )
                    else:
                        yhat_ref = run_reference_streaming(
                            dim,
                            kernel,
                            eps_ref,
                            l2scaled,
                            n_train=N,
                            seed=dim + N,
                            x_trg=x_trg,
                            y_trg_test=y_trg_test,
                            cg_tol=eps_ref,
                            maxiter=CG_MAXITER,
                            chunk_size=TRAIN_CHUNK_SIZE,
                            x_train_cached=x_train,
                        )
                else:
                    yhat_ref = run_reference(
                        dim, kernel, eps_ref, l2scaled, x_train, y_train, x_trg, y_trg_test, eps_ref, CG_MAXITER
                    )
                err_eepm = rms(yhat, yhat_ref)
                err_rmse_ex = rms(y_trg_test, yhat_ref)
            else:
                yhat_ref = None
                err_eepm = float("nan")
                err_rmse_ex = float("nan")

            pre = info["pre_times"]["time_precompute_total"]
            eig = 0.0
            precond = 0.0
            solve = info["solve_time"]
            mean = info["mean_time"]
            total = pre + eig + precond + solve + mean

            row = {
                "N": N,
                "dim": dim,
                "kernel": kernel_label,
                "eps": eps,
                "m": info["m"],
                "pre": pre,
                "eig": eig,
                "precond": precond,
                "solve": solve,
                "mean": mean,
                "total": total,
                "iters": info["iters"],
                "eepm": err_eepm,
                "rms": err_rms,
                "rmse_ex": err_rmse_ex,
            }
            results.append(row)
            cache[(kernel_name, dim, N)] = {
                "N": N,
                "kernel": kernel,
                "eps": eps,
                "l2scaled": l2scaled,
                "x_train": x_train,
                "y_train": y_train,
                "x_trg": x_trg,
                "y_trg_test": y_trg_test,
                "yhat": yhat,
                "yhat_ref": yhat_ref,
                "info": info,
            }

            print_table_row(
                N,
                dim,
                kernel_label,
                eps,
                info["m"],
                pre,
                eig,
                precond,
                solve,
                mean,
                total,
                info["iters"],
                err_eepm,
                err_rms,
                err_rmse_ex,
            )


N d kernel eps m pre_s eig_s precond_s solve_s mean_s total_s iters eepm rms rmse_ex
    1000 1      SE  1.0e-04   13    0.108    0.000    0.000    0.007    0.004    0.119     18       nan  3.00e-01       nan
  100000 1      SE  1.0e-04   13    0.010    0.000    0.000    0.002    0.005    0.016     14       nan  3.03e-01       nan
10000000 1      SE  1.0e-04   13    0.256    0.000    0.000    0.002    0.124    0.383     19       nan  3.04e-01       nan
    1000 2      SE  1.0e-04   16    0.010    0.000    0.000    0.014    0.004    0.028     62       nan  3.17e-01       nan
  100000 2      SE  1.0e-04   16    0.023    0.000    0.000    0.035    0.007    0.065    192       nan  2.97e-01       nan
10000000 2      SE  1.0e-04   16    1.407    0.000    0.000    0.026    0.467    1.899    136       nan  2.96e-01       nan
    1000 3      SE  1.0e-03   17    0.053    0.000    0.000    0.614    0.025    0.693     33       nan  4.02e-01       nan
  100000 3      SE  1.0e-03   17    0.058    0.

In [27]:
def kernel_label_from_spec(kernel):
    fam = kernel.fam.lower()
    if fam in ("squared-exponential", "squared_exponential", "se"):
        return "SE"
    if fam == "matern" and kernel.nu == 0.5:
        return "Mat12"
    return kernel.fam


def topq_table_header() -> str:
    return "top_q N d kernel eps m pre_s eig_s precond_s solve_s mean_s total_s iters eepm rms rmse_ex"


def format_topq_row(row: dict) -> str:
    return (
        f"{row['top_q']:>5d} {row['N']:>8d} {row['dim']:>1d} {row['kernel']:>7s} {row['eps']:>8.1e} "
        f"{row['m']:>4d} {row['pre']:>8.3f} {row['eig']:>8.3f} {row['precond']:>8.3f} {row['solve']:>8.3f} "
        f"{row['mean']:>8.3f} {row['total']:>8.3f} {row['iters']:>6d} {row['eepm']:>9.2e} "
        f"{row['rms']:>9.2e} {row['rmse_ex']:>9.2e}"
    )


def topq_sweep_for_case(cache_key, top_q):
    entry = cache[cache_key]
    kernel = entry["kernel"]
    eps = entry["eps"]
    x_trg = entry["x_trg"]
    y_trg_test = entry["y_trg_test"]
    yhat_ref = entry["yhat_ref"]
    state = entry["info"]["state"]
    compute_eepm = COMPUTE_EEPM and yhat_ref is not None

    nufft_eps = eps / 10.0
    solver = EFGPSolver(kernel, reg_lambda=SIGMA_TRUE ** 2, eps=eps, nufft_tol=nufft_eps)
    matvec = lambda v: solver._apply_A(state, v)

    pre_base = entry["info"]["pre_times"]["time_precompute_total"]

    precond, _eigpairs, _mu, t_eig, t_precond, top_q_used = bm.build_precond_from_state(
        solver,
        state,
        top_q,
        eig_method=EIG_METHOD,
        eig_tol=EIG_TOL,
        eig_maxiter=EIG_MAXITER,
        eig_block_size=EIG_BLOCK_SIZE,
        eig_oversample=EIG_OVERSAMPLE,
    )
    precond_apply = precond.apply if precond is not None else None

    t_s0 = time.perf_counter()
    beta, it, _relres, _hist = bm.pcg_solve(
        matvec, state.rhs, tol=eps, maxiter=CG_MAXITER, precond=precond_apply, store_history=False
    )
    t_s1 = time.perf_counter()

    x_eval = x_trg
    if MEAN_INCLUDE_TRAIN:
        x_train = entry.get("x_train")
        if x_train is None:
            raise ValueError("cache entry missing x_train for MEAN_INCLUDE_TRAIN=True")
        x_eval = np.vstack([x_train, x_trg])
    t_m0 = time.perf_counter()
    yhat_all = solver.predict(x_eval, beta, state)
    t_m1 = time.perf_counter()

    yhat = yhat_all[-x_trg.shape[0]:] if MEAN_INCLUDE_TRAIN else yhat_all
    err_rms = rms(y_trg_test, yhat)
    if compute_eepm:
        err_eepm = rms(yhat, yhat_ref)
        err_rmse_ex = rms(y_trg_test, yhat_ref)
    else:
        err_eepm = float("nan")
        err_rmse_ex = float("nan")

    mean_time = t_m1 - t_m0
    solve_time = t_s1 - t_s0
    pre = pre_base
    eig = t_eig
    precond = t_precond
    total = pre + eig + precond + solve_time + mean_time
    m = (state.grid.mtot - 1) // 2

    n_value = entry.get("N")
    if n_value is None and entry.get("x_train") is not None:
        n_value = int(entry["x_train"].shape[0])
    if n_value is None:
        raise ValueError("cache entry missing N")

    row = {
        "top_q": int(top_q_used),
        "N": int(n_value),
        "dim": int(kernel.dim),
        "kernel": kernel_label_from_spec(kernel),
        "eps": float(eps),
        "m": int(m),
        "pre": float(pre),
        "eig": float(eig),
        "precond": float(precond),
        "solve": float(solve_time),
        "mean": float(mean_time),
        "total": float(total),
        "iters": int(it),
        "eepm": float(err_eepm),
        "rms": float(err_rms),
        "rmse_ex": float(err_rmse_ex),
    }
    return row


def _n_list_from_cache(kernel_name: str, dim: int) -> list[int]:
    return sorted({key[2] for key in cache.keys() if key[0] == kernel_name and key[1] == dim})


def _write_csv(rows: list[dict], path: Path):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def topq_sweep_table(
    top_q_list,
    kernel_names=None,
    dims=None,
    print_rows: bool = True,
    csv_path: Path | None = None,
):
    if kernel_names is None:
        kernel_names = ["se", "matern"]
    if dims is None:
        dims = DIM_LIST

    rows_by_q = {}
    all_rows = []
    for top_q in top_q_list:
        rows = []
        if print_rows:
            print(f"\nTOP_Q={top_q}")
            print(topq_table_header())

        for kernel_name in kernel_names:
            for dim in dims:
                n_list = _n_list_from_cache(kernel_name, dim)
                if not n_list:
                    n_list = [10 ** k for k in N_EXP_LIST]
                    missing = [n for n in n_list if (kernel_name, dim, n) not in cache]
                    if missing:
                        raise ValueError(
                            "cache missing cases; run Cell 3 first or align N_EXP_LIST. "
                            f"missing={[(kernel_name, dim, n) for n in missing]}"
                        )
                for N in n_list:
                    key = (kernel_name, dim, N)
                    row = topq_sweep_for_case(key, top_q)
                    rows.append(row)
                    all_rows.append(row)
                    if print_rows:
                        print(format_topq_row(row))
        rows_by_q[top_q] = rows
    if csv_path is not None:
        _write_csv(all_rows, csv_path)
    return rows_by_q


# 运行：
# rows_by_q = topq_sweep_table(TOP_Q_SWEEP)
# 只跑 top_q=0：
# rows_q0 = topq_sweep_table([0])[0]
# 制表（可选）：
# import pandas as pd
# df_q0 = pd.DataFrame(rows_q0)
# df_q0

# 扫描 q 的表（含 eepm/rmse_ex 列，未计算时为 nan）
rows_by_q = topq_sweep_table(TOP_Q_SWEEP, csv_path=CSV_PATH_TABLE2_TOPQ)



TOP_Q=0
top_q N d kernel eps m pre_s eig_s precond_s solve_s mean_s total_s iters eepm rms rmse_ex
    0     1000 1      SE  1.0e-04   13    0.108    0.000    0.000    0.003    0.005    0.116     18       nan  3.00e-01       nan
    0   100000 1      SE  1.0e-04   13    0.010    0.000    0.000    0.002    0.005    0.016     14       nan  3.03e-01       nan
    0 10000000 1      SE  1.0e-04   13    0.256    0.000    0.000    0.002    0.128    0.386     19       nan  3.04e-01       nan
    0     1000 2      SE  1.0e-04   16    0.010    0.000    0.000    0.013    0.005    0.028     62       nan  3.17e-01       nan
    0   100000 2      SE  1.0e-04   16    0.023    0.000    0.000    0.042    0.007    0.072    192       nan  2.97e-01       nan
    0 10000000 2      SE  1.0e-04   16    1.407    0.000    0.000    0.025    0.521    1.953    136       nan  2.96e-01       nan
    0     1000 3      SE  1.0e-03   17    0.053    0.000    0.000    0.597    0.041    0.691     33       nan  4.02e-01 

In [28]:
def topq_sweep_all(kernel_name: str, dim: int, top_q_list):
    n_list = list(N_LIST)
    for N in n_list:
        key = (kernel_name, dim, N)
        print("\ncase=", key)
        topq_sweep_for_case(key, top_q_list)


# dims sweep helper
# for dim in DIM_LIST:
#     topq_sweep_all("se", dim, TOP_Q_SWEEP)


# example:
# topq_sweep_all("se", 2, TOP_Q_SWEEP)
